# Test tool for wikipedia API

In [6]:
from pydantic_ai import Agent
from pydantic_ai.models import ModelSettings
from model import nvidia_model

music_agent = Agent(
    model=nvidia_model("nvidia/nemotron-3.5-super-vl-120b-a12b"),
    instructions=("You are a music quiz master. Your job is to come up with fun and unexpected follow-up questions"),
        model_settings=ModelSettings(thinking="minimal")
)

result = await music_agent.run("give me a followup qustion on a Rihanna song from 2000")

print(result.output)

Here’s a fun, unexpected follow-up if your last question was about Rihanna’s 2006 hit **“SOS”**:

> **“Rihanna’s ‘SOS’ is built around one of the most recognizable samples in pop history. Which 1980s synth-pop classic—originally recorded by Gloria Jones in 1965 but turned into a worldwide smash by Soft Cell in 1981—provides the song’s pulsing hook?**”

**Answer:** “Tainted Love.”

It’s a great curveball because most people don’t associate early-2000s Rihanna with an 80s new-wave staple.


In [9]:
"""Tools that the music agent can call."""

import os

import httpx
from pydantic_ai import RunContext

from .deps import MusicData

# Wikimedia requires a User-Agent in the format:
# AppName/version (contact) library/version
# Otherwise the API answers 403.
CONTACT = os.environ.get("WIKI_CONTACT")
CONTACT_TEXT = "https://github.com/mirandalyng/examensarbete"
if CONTACT:
    CONTACT_TEXT += f"; {CONTACT}"

HEADERS = {
    "User-Agent": f"MusicQuizExamensarbete/0.1 ({CONTACT_TEXT}) httpx/{httpx.__version__}"
}


async def get_artist_wikipedia(ctx: RunContext[MusicData]) -> dict:
    """Fetch a short Wikipedia summary about the artist of the current song.

    Call this before writing the follow-up question. Returns the summary text
    and the url of the Wikipedia page. The text is empty if no page was found.
    """
    artist = ctx.deps.artist_names.split(", ")[0]

    async with httpx.AsyncClient(headers=HEADERS, timeout=10, follow_redirects=True) as client:
        # 1. Search for the artist's page
        search = await client.get(
            "https://en.wikipedia.org/w/api.php",
            params={
                "action": "query",
                "list": "search",
                "format": "json",
                "srsearch": f"{artist} music",
                "srlimit": 1,
            },
        )
        search.raise_for_status()
        hits = search.json()["query"]["search"]
        if not hits:
            return {"text": "", "url": None}

        # 2. Fetch the summary of the best hit
        title = hits[0]["title"].replace(" ", "_")
        page = await client.get(f"https://en.wikipedia.org/api/rest_v1/page/summary/{title}")
        page.raise_for_status()
        data = page.json()

    return {
        "text": data.get("extract", ""),
        "url": data.get("content_urls", {}).get("desktop", {}).get("page"),
    }

ImportError: attempted relative import with no known parent package

In [8]:
import httpx
from agent.wikipedia import HEADERS

response = httpx.get(
    "https://en.wikipedia.org/w/api.php", 
    params={"action": "query",
            "list": "search",
            "format": "json",
            "srsearch": "abba music",
            "srlimit": 1}, 
    headers=HEADERS, follow_redirects=True)

print(response.status_code)

ModuleNotFoundError: No module named 'agent.wikipedia'